In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [3]:
load_dotenv()
model = ChatGoogleGenerativeAI(model='gemini-2.0-flash-lite')

In [4]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [5]:
s_m = model.with_structured_output(EvaluationSchema)

In [ ]:
essay = """
In the 21st century, technology has revolutionized almost every aspect of human life, and education is no exception. From online classes to digital textbooks, technology has transformed the way students learn, teachers teach, and institutions operate. While this transformation has brought remarkable benefits, it also presents challenges that must be addressed to ensure equitable and meaningful learning experiences.

One of the most significant advantages of technology in education is accessibility. Students no longer need to be physically present in classrooms to receive quality education. Online learning platforms such as Coursera, Khan Academy, and Google Classroom allow learners from all corners of the world to access educational materials. This democratization of education has empowered millions who might otherwise lack access due to geographical or financial limitations.

Moreover, technology makes learning more engaging and personalized. Interactive tools such as simulations, videos, and gamified lessons help students grasp complex concepts more easily. Artificial intelligence can also analyze student performance and adapt content to suit individual learning styles, making education more efficient and effective than ever before.

However, the growing reliance on technology in education also poses challenges. The digital divide remains a major concern, as not all students have access to reliable internet or devices. This inequality can widen the gap between privileged and underprivileged learners. Additionally, excessive screen time can impact students’ mental and physical health, while overreliance on digital tools may reduce critical thinking and interpersonal skills.

In conclusion, technology has undoubtedly reshaped modern education for the better by increasing accessibility and improving learning methods. Yet, it is essential to use these tools responsibly and ensure equal access for all students. By balancing innovation with inclusivity, technology can continue to enhance education and prepare future generations for an increasingly digital world.
"""

In [7]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'

In [9]:
s_m.invoke(prompt).score

8

In [28]:
class KCState(TypedDict):

    essay: str
    lang_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[int], operator.add]
    avg_score: float

In [29]:
def evaluate_lang(state: KCState):
    
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output = s_m.invoke(prompt)

    return {'lang_feedback': output.feedback, 'individual_score': [output.score]}

In [30]:
def evaluate_analysis(state: KCState):
    
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output = s_m.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_score': [output.score]}

In [31]:
def evaluate_thought(state: KCState):
    
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output = s_m.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_score': [output.score]}

In [32]:
def final_evaluation(state: KCState):
    
    # summary feedback
     prompt = (
        f"Based on the following feedbacks create a summarized feedback:\n"
        f"Language feedback: {state['lang_feedback']}\n"
        f"Depth of analysis feedback: {state['analysis_feedback']}\n"
        f"Clarity of thought feedback: {state['clarity_feedback']}"
    )


     overall_feedback = model .invoke(prompt).content

    # avg calc
     avg_score = sum(state['individual_score'])/len(state['individual_score'])

     return {'overall_feedback': overall_feedback, 'avg_score': avg_score}


In [33]:
graph = StateGraph(KCState)

graph.add_node("evaluate_lang", evaluate_lang)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)

# edges
graph.add_edge(START, 'evaluate_lang')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')
graph.add_edge('evaluate_lang', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')
graph.add_edge('final_evaluation', END)

workflow = graph.compile()


In [36]:
initial_state = {
    'essay': essay
}

final_state = workflow.invoke(initial_state)
print(final_state)

{'essay': '\nIn the 21st century, technology has revolutionized almost every aspect of human life, and education is no exception. From online classes to digital textbooks, technology has transformed the way students learn, teachers teach, and institutions operate. While this transformation has brought remarkable benefits, it also presents challenges that must be addressed to ensure equitable and meaningful learning experiences.\n\nOne of the most significant advantages of technology in education is accessibility. Students no longer need to be physically present in classrooms to receive quality education. Online learning platforms such as Coursera, Khan Academy, and Google Classroom allow learners from all corners of the world to access educational materials. This democratization of education has empowered millions who might otherwise lack access due to geographical or financial limitations.\n\nMoreover, technology makes learning more engaging and personalized. Interactive tools such as